# Arm B: Auditing C4 Exclusions with a Neural Toxicity Classifier

This notebook runs **Arm B of the Excluded Data Audit**. Instead of relying solely on Google C4's crude
keyword blocklist (LDNOOBW), this experiment evaluates the exact same web pages side-by-side using an
actual context-aware neural toxicity classifier (`unitary/toxic-bert`).

### Core Research Questions:
1. When using a context-aware toxicity classifier, does the **$83\%$ exclusion penalty on LGBTQ+ terms** (like "lesbian" and "gay") disappear?
2. Does the **$5\\times$ higher censorship rate on African American English (AAE)** decrease?
3. How much of the excluded text was actually toxic vs. innocent collateral damage?

**Recommended Settings:**
- Accelerator: **GPU T4 x2** (runs batched PyTorch in ~15–20 minutes) or standard CPU (~1.5 hours).
- Internet: **ON** (to download `unitary/toxic-bert` model weights and C4 shards).
- Run mode: Click **Save Version -> Save & Run All (Commit)** and close the tab!
- Output: Download `f5_arm_b_toxicity_results.zip` when complete.


## 1. Setup: Copy Dataset into Writable Directory


In [ ]:
%%bash
SRC=$(dirname "$(dirname "$(find /kaggle/input -path '*scripts/scan_noblocklist.py' -o -path '*scripts/scan_toxicity.py' | head -1)")")
echo "Project source directory: $SRC"
test -f "$SRC/data/ldnoobw_en_25e679f.txt" || { echo "LDNOOBW blocklist not found!"; exit 1; }

mkdir -p /kaggle/working/f5_toxicity/scripts /kaggle/working/f5_toxicity/data /kaggle/working/f5_toxicity/results/arm_b_toxicity_4_shards
cp -rn "$SRC/data" /kaggle/working/f5_toxicity/
pip install -q transformers huggingface_hub
nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Running on CPU"


## 2. Embedded Scripts (Always Runs Current Repo Code)


In [ ]:
%%writefile /kaggle/working/f5_toxicity/scripts/dialect.py
"""Blodgett, Green and O'Connor (2016) demographic dialect model ("TwitterAAE"), ported to Python 3.

Dodge et al. 2021 §5.3 use it to give each document a posterior over four dialect "topics" and assign the argmax.
The model files come from github.com/slanglab/twitteraae at a pinned commit (MIT licence) and are downloaded once.

Two inference routines:
- exact(): a line-by-line port of the repo's predict.py (collapsed variational Bayes, CVB0, sequential token
  updates, alpha=1, 5 passes; None if no token, or fewer than 20% of tokens, are in the vocabulary).
- fast(): the same model and gates, but each pass updates all tokens at once (Jacobi instead of Gauss-Seidel)
  and identical words are updated together. It is about 100x faster, which a free-tier CPU needs for millions of
  documents. scan_noblocklist.py runs both on the first documents of every shard and records how often the
  argmax agrees, so the approximation is measured, not assumed.

Assumptions (the paper does not say): the whole document is scored; tokens are lowercased runs of letters and
digits with internal apostrophes (the repo used a tweet tokenizer, twokenize, on tweets); documents the model
cannot score are reported as "unassigned".
"""
import os, re, urllib.request

import numpy as np

SHA = "db4d6d126ca26b9bb9a70d7ce18bdd5833077523"
URL = f"https://raw.githubusercontent.com/slanglab/twitteraae/{SHA}/model/"
FILES = ["model_count_table.txt", "model_vocab.txt"]
# repo order: African-American, Hispanic, Asian, White. Dodge et al. call the third "other"; the repo README notes
# that only the AA and White topics were validated.
LABELS = ["AAE", "Hispanic", "Other", "WAE"]
TOKEN = re.compile(r"[a-z0-9]+(?:'[a-z]+)*")


def ensure_files(model_dir):
    os.makedirs(model_dir, exist_ok=True)
    for f in FILES:
        p = os.path.join(model_dir, f)
        if not os.path.exists(p):
            urllib.request.urlretrieve(URL + f, p + ".tmp")
            os.replace(p + ".tmp", p)


class DialectModel:
    def __init__(self, model_dir):
        ensure_files(model_dir)
        n_wk = np.loadtxt(os.path.join(model_dir, "model_count_table.txt"))
        self.wordprobs = (n_wk + 1) / n_wk.sum(0)             # as in predict.py
        with open(os.path.join(model_dir, "model_vocab.txt"), encoding="utf-8") as f:
            vocab = [line.split("\t")[-1].strip() for line in f]
        assert len(vocab) == n_wk.shape[0]
        self.w2num = {w: i for i, w in enumerate(vocab)}
        self.K = n_wk.shape[1]

    def tokens(self, text):
        return TOKEN.findall(text.lower())

    def _gate(self, toks, thresh1, thresh2):
        ids = [self.w2num[w] for w in toks if w in self.w2num]
        if len(ids) < thresh1 or len(ids) / max(1, len(toks)) < thresh2:
            return None
        return ids

    def exact(self, toks, alpha=1, numpasses=5, thresh1=1, thresh2=0.2):
        """Port of predict.predict / infer_cvb0 (sequential updates)."""
        ids = self._gate(toks, thresh1, thresh2)
        if ids is None:
            return None
        qs = self.wordprobs[ids].copy()
        qs /= qs.sum(1, keepdims=True)
        lik = qs.copy()
        q_k = qs.sum(0)
        for _ in range(1, numpasses):
            for i in range(len(ids)):
                q_k -= qs[i]
                qs[i] = lik[i] * (q_k + alpha)
                qs[i] /= qs[i].sum()
                q_k += qs[i]
        return q_k / q_k.sum()

    def fast(self, toks, alpha=1, numpasses=5, thresh1=1, thresh2=0.2):
        """Same model and gates; all tokens updated at once per pass, identical words grouped."""
        ids = self._gate(toks, thresh1, thresh2)
        if ids is None:
            return None
        u, c = np.unique(ids, return_counts=True)
        lik = self.wordprobs[u]
        lik = lik / lik.sum(1, keepdims=True)
        qs = lik.copy()
        q_k = c @ qs
        for _ in range(1, numpasses):
            qs = lik * (q_k - qs + alpha)
            qs /= qs.sum(1, keepdims=True)
            q_k = c @ qs
        return q_k / q_k.sum()


In [ ]:
%%writefile /kaggle/working/f5_toxicity/scripts/scan_toxicity.py
"""Arm B of Dodge et al. 2021 §5: audit C4 filtering using a neural toxicity classifier.

Instead of a crude keyword blocklist (LDNOOBW), this script evaluates web text with an actual
context-aware toxicity classification model (e.g., unitary/toxic-bert) side-by-side with the
original LDNOOBW regex filter on the exact same documents.

For every document in C4.EN.NOBLOCKLIST, it records:
1. Excluded by Blocklist: matches LDNOOBW regex.
2. Excluded by Toxicity Classifier: model toxicity probability >= threshold (default 0.5).
3. Identity terms: matches any of the 22 identity regexes (Table 6).
4. Dialect: predicted by the Blodgett et al. TwitterAAE model (AAE, Hispanic, Other, WAE).
5. Filter agreement: both kept, both excluded, blocklist-only excluded, toxicity-only excluded.

Counts are aggregated per web domain (URL host) and saved as counts_<shard>.npz for domain bootstrapping.

Usage:
    # On Kaggle with GPU (T4 x2 or single T4):
    python scripts/scan_toxicity.py --n 4 --seed 0 --batch_size 128 --out_dir results/arm_b_toxicity_4_shards
    # On CPU:
    python scripts/scan_toxicity.py --n 1 --seed 0 --batch_size 32 --limit 50000 --out_dir results/test_toxicity
"""
import argparse, gzip, json, os, random, re, time
from collections import defaultdict
from urllib.parse import urlparse

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
BADWORDS = os.path.join(ROOT, "data", "ldnoobw_en_25e679f.txt")
MODEL_DIR = os.path.join(ROOT, "data", "twitteraae_model")
N_SHARDS = 1024

IDENTITY = [
    ("homosexual", r"homosexuals?"), ("gay", r"gays?"), ("non-binary", r"non[ -]?binary"),
    ("trans", r"trans(?:|\+|gender)"), ("lesbian", r"lesbians?"), ("black", r"blacks?"),
    ("african american", r"african[ -]americans?"), ("latina / latino", r"latin[oax]s?"),
    ("asian", r"asian(?:[ -]american)?s?"), ("muslim", r"muslims?"), ("jewish", r"jew(?:|s|ish)"),
    ("women", r"wom[ae]n"), ("female", r"females?"), ("man", r"m[ae]n"), ("male", r"males?"),
    ("straight", r"straights?"), ("heterosexual", r"heterosexuals?"), ("bisexual", r"bi-?sexuals?"),
    ("white", r"whites?"), ("caucasian", r"caucasians?"), ("european", r"european(?:[ -]american)?s?"),
    ("christian", r"christians?"),
]
DIALECTS = ["AAE", "Hispanic", "Other", "WAE", "unassigned"]

# Columns stored per host in the count matrix
COLUMNS = (
    ["docs", "ex_blocklist", "ex_toxic", "both_ex", "blocklist_only", "toxic_only", "both_kept"]
    + [f"bl:{t}|{k}" for t, _ in IDENTITY for k in ("docs", "excluded")]
    + [f"tx:{t}|{k}" for t, _ in IDENTITY for k in ("docs", "excluded")]
    + [f"bl:dialect:{d}|{k}" for d in DIALECTS for k in ("docs", "excluded")]
    + [f"tx:dialect:{d}|{k}" for d in DIALECTS for k in ("docs", "excluded")]
)


def badwords_regex():
    words = [w.strip() for w in open(BADWORDS, encoding="utf-8") if w.strip()]
    return re.compile(r"(?:\W|^)({})(?:\W|$)".format("|".join(words)))


def identity_regex():
    return re.compile("|".join(rf"\b(?P<g{i}>{p})\b" for i, (_, p) in enumerate(IDENTITY)), re.I)


def shard_name(i):
    return f"c4-train.{i:05d}-of-{N_SHARDS:05d}"


class BatchedToxicityClassifier:
    def __init__(self, model_name, device, max_length=256):
        self.device = device
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.model.to(device)
        self.model.eval()
        self.is_multilabel = getattr(self.model.config, "problem_type", None) == "multi_label_classification" or \
                             self.model.config.num_labels > 2

    @torch.no_grad()
    def score_batch(self, texts):
        if not texts:
            return []
        inputs = self.tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        ).to(self.device)

        use_amp = self.device.type == "cuda"
        with torch.amp.autocast(device_type=self.device.type, enabled=use_amp):
            logits = self.model(**inputs).logits

        if self.is_multilabel:
            # Multi-label classification (e.g., toxic, severe_toxic, obscene, threat, insult, identity_hate)
            probs = torch.sigmoid(logits)
            # Max toxicity score across any category (or specifically primary 'toxic' column index 0)
            scores = probs.max(dim=-1).values.cpu().numpy()
        else:
            # Binary classification (label 1 = toxic)
            probs = torch.softmax(logits, dim=-1)
            scores = probs[:, 1].cpu().numpy()

        return scores.tolist()


def scan_file(path, shard, tox_clf, args):
    from dialect import DialectModel
    bad, ident = badwords_regex(), identity_regex()
    dialect_model = DialectModel(MODEL_DIR)
    col = {c: i for i, c in enumerate(COLUMNS)}
    per_host = defaultdict(lambda: np.zeros(len(COLUMNS), dtype=np.int64))

    discrepancies = []
    n = n_ex_bl = n_ex_tx = 0
    batch_docs = []

    def flush_batch(batch):
        nonlocal n, n_ex_bl, n_ex_tx
        texts = [d["text"] for d in batch]
        scores = tox_clf.score_batch(texts)

        for d, tx_score in zip(batch, scores):
            text, url = d["text"], d["url"]
            n += 1
            ex_bl = bool(bad.search(text.lower()))
            ex_tx = bool(tx_score >= args.threshold)

            n_ex_bl += ex_bl
            n_ex_tx += ex_tx

            host = urlparse(url).netloc
            row = per_host[host]

            row[col["docs"]] += 1
            row[col["ex_blocklist"]] += ex_bl
            row[col["ex_toxic"]] += ex_tx

            if ex_bl and ex_tx:
                row[col["both_ex"]] += 1
            elif ex_bl and not ex_tx:
                row[col["blocklist_only"]] += 1
            elif not ex_bl and ex_tx:
                row[col["toxic_only"]] += 1
            else:
                row[col["both_kept"]] += 1

            # Identity term hits
            hit = {int(m.lastgroup[1:]) for m in ident.finditer(text)}
            for i in hit:
                term_name = IDENTITY[i][0]
                row[col[f"bl:{term_name}|docs"]] += 1
                row[col[f"bl:{term_name}|excluded"]] += ex_bl
                row[col[f"tx:{term_name}|docs"]] += 1
                row[col[f"tx:{term_name}|excluded"]] += ex_tx

            # Dialect
            toks = dialect_model.tokens(text)
            post = dialect_model.fast(toks)
            dl = DIALECTS[int(np.argmax(post))] if post is not None else "unassigned"

            row[col[f"bl:dialect:{dl}|docs"]] += 1
            row[col[f"bl:dialect:{dl}|excluded"]] += ex_bl
            row[col[f"tx:dialect:{dl}|docs"]] += 1
            row[col[f"tx:dialect:{dl}|excluded"]] += ex_tx

            # Collect informative discrepancies for qualitative audit
            if ex_bl != ex_tx and len(discrepancies) < args.sample_discrepancies:
                discrepancies.append({
                    "url": url,
                    "ex_blocklist": ex_bl,
                    "ex_toxic": ex_tx,
                    "toxic_score": round(float(tx_score), 4),
                    "dialect": dl,
                    "identity_hits": [IDENTITY[i][0] for i in hit],
                    "excerpt": text[:500]
                })

    with gzip.open(path, "rt", encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            batch_docs.append(d)
            if len(batch_docs) >= args.batch_size:
                flush_batch(batch_docs)
                batch_docs = []
            if args.limit and n >= args.limit:
                break
        if batch_docs:
            flush_batch(batch_docs)

    hosts = list(per_host)
    counts_arr = np.stack([per_host[h] for h in hosts]) if hosts else np.zeros((0, len(COLUMNS)), dtype=np.int64)

    np.savez_compressed(
        os.path.join(args.out_dir, f"counts_{shard}.npz"),
        hosts=np.array(hosts, dtype=str),
        counts=counts_arr,
        columns=np.array(COLUMNS)
    )

    if discrepancies:
        disc_path = os.path.join(args.out_dir, f"discrepancies_{shard}.jsonl.gz")
        with gzip.open(disc_path, "wt", encoding="utf-8") as g:
            for r in discrepancies:
                g.write(json.dumps(r, ensure_ascii=False) + "\n")

    stats = {
        "shard": shard,
        "documents": n,
        "ex_blocklist": n_ex_bl,
        "ex_blocklist_rate": n_ex_bl / max(1, n),
        "ex_toxic": n_ex_tx,
        "ex_toxic_rate": n_ex_tx / max(1, n),
        "hosts": len(hosts),
        "discrepancies_sampled": len(discrepancies)
    }

    with open(os.path.join(args.out_dir, f"stats_{shard}.json"), "w") as f:
        json.dump(stats, f, indent=1)

    return stats


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model", default="unitary/toxic-bert", help="HuggingFace toxicity classification model")
    ap.add_argument("--threshold", type=float, default=0.5, help="Toxicity score threshold for exclusion")
    ap.add_argument("--shards", type=int, nargs="*")
    ap.add_argument("--range", type=int, nargs=2, metavar=("START", "END"))
    ap.add_argument("--n", type=int, default=4, help="Number of seeded shards to scan")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--files", nargs="*", help="Local noblocklist shard files")
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--limit", type=int, default=None, help="Max documents per shard (useful for quick tests)")
    ap.add_argument("--max_length", type=int, default=256)
    ap.add_argument("--sample_discrepancies", type=int, default=1000)
    ap.add_argument("--keep_shards", action="store_true")
    ap.add_argument("--cache_dir", default=None)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Loading toxicity model: {args.model} on {device} ...", flush=True)
    tox_clf = BatchedToxicityClassifier(args.model, device, max_length=args.max_length)

    from dialect import ensure_files
    ensure_files(MODEL_DIR)

    if args.files:
        jobs = [(re.search(r"c4-train\.\d{5}-of-\d{5}", os.path.basename(p)).group(0), p) for p in args.files]
    else:
        if args.shards:
            idx = args.shards
        elif args.range:
            idx = list(range(max(0, args.range[0]), min(N_SHARDS, args.range[1])))
        else:
            idx = sorted(random.Random(args.seed).sample(range(N_SHARDS), args.n))
        jobs = [(shard_name(i), None) for i in idx]

    meta_path = os.path.join(args.out_dir, "shards.json")
    meta = json.load(open(meta_path)) if os.path.exists(meta_path) else {"shards": []}
    meta["shards"] = sorted(set(meta["shards"]) | {s for s, _ in jobs})
    meta["config"] = "en.noblocklist"
    meta["model"] = args.model
    meta["threshold"] = args.threshold
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=1)

    print(f"Processing {len(jobs)} shards with {args.model} (threshold={args.threshold}) ...", flush=True)
    for shard, path in jobs:
        done_stat = os.path.join(args.out_dir, f"stats_{shard}.json")
        if os.path.exists(done_stat):
            s = json.load(open(done_stat))
            print(f"Skipping completed shard {shard} ({s['documents']:,} docs)", flush=True)
            continue

        t0 = time.time()
        downloaded = False
        if path is None:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("allenai/c4", f"en.noblocklist/{shard}.json.gz", repo_type="dataset",
                                   cache_dir=args.cache_dir)
            downloaded = True

        s = scan_file(path, shard, tox_clf, args)
        if downloaded and not args.keep_shards:
            os.remove(os.path.realpath(path))

        elapsed = time.time() - t0
        print(f"[{shard}] {s['documents']:,} docs in {elapsed:.0f}s | "
              f"Blocklist Excluded: {100 * s['ex_blocklist_rate']:.2f}% | "
              f"Toxicity Excluded: {100 * s['ex_toxic_rate']:.2f}%", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/f5_toxicity/scripts/analyze_toxicity.py
"""Arm B analysis: compare Blocklist (LDNOOBW) vs Toxicity Classifier (e.g. Toxic-BERT).

Reads count matrices generated by scan_toxicity.py across shards, aggregates by web domain,
computes 95% bootstrap confidence intervals, and outputs comparative tables and publication-ready figures.

Outputs in --out_dir:
- summary.md: overall metrics, filter agreement/discrepancy matrix
- pmi_comparison.csv & fig_pmi_comparison.png: 22 identity terms PMI and removal rates
- dialects_comparison.csv & fig_dialects_comparison.png: AAE, Hispanic, Other, WAE removal rates
- discrepancies.csv: qualitative samples of false alarms vs stealth toxicity

Usage:
    python scripts/analyze_toxicity.py --scan results/arm_b_toxicity_4_shards --out_dir results/arm_b_toxicity_4_shards/analysis
"""
import argparse, glob, gzip, json, os

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)

IDENTITY = [
    ("homosexual", "sexual orientation"), ("gay", "sexual orientation"), ("non-binary", "gender identity"),
    ("trans", "gender identity"), ("lesbian", "sexual orientation"), ("black", "race / ethnicity"),
    ("african american", "race / ethnicity"), ("latina / latino", "race / ethnicity"),
    ("asian", "race / ethnicity"), ("muslim", "religion"), ("jewish", "religion"),
    ("women", "gender identity"), ("female", "gender identity"), ("man", "gender identity"),
    ("male", "gender identity"), ("straight", "sexual orientation"), ("heterosexual", "sexual orientation"),
    ("bisexual", "sexual orientation"), ("white", "race / ethnicity"), ("caucasian", "race / ethnicity"),
    ("european", "race / ethnicity"), ("christian", "religion"),
]
DIALECTS = ["AAE", "Hispanic", "Other", "WAE", "unassigned"]

# Reference palette
C_BL = "#e34948"       # Blocklist (Red)
C_TX = "#2a78d6"       # Toxicity Classifier (Blue)
C_AGREE = "#1baf7a"    # Agreement (Green)
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"


def load_counts(scan_dir):
    files = sorted(glob.glob(os.path.join(scan_dir, "stats_*.json")))
    if not files:
        raise SystemExit(f"No finished shards (stats_*.json) found in {scan_dir}")
    stats = [json.load(open(p)) for p in files]
    frames, columns = [], None
    for s in stats:
        npz_path = os.path.join(scan_dir, f"counts_{s['shard']}.npz")
        if os.path.exists(npz_path):
            z = np.load(npz_path)
            columns = list(z["columns"])
            frames.append(pd.DataFrame(z["counts"], index=z["hosts"], columns=columns))
    df = pd.concat(frames).groupby(level=0).sum()
    return df, columns, stats


def bootstrap(X, B, seed, batch=10):
    from scipy import sparse
    Xs = sparse.csr_matrix(X.astype(np.float64)).T.tocsr()
    rng = np.random.default_rng(seed)
    out = []
    for b0 in range(0, B, batch):
        W = rng.poisson(1.0, size=(X.shape[0], min(batch, B - b0))).astype(np.float32)
        out.append(np.asarray(Xs @ W).T)
    return np.concatenate(out)


def ci(a):
    return np.nanpercentile(a, [2.5, 97.5], axis=0)


def style(ax):
    ax.set_facecolor(SURFACE)
    for s in ("top", "right", "left"):
        ax.spines[s].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK2, length=0)
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scan", required=True, help="Folder containing counts_*.npz")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--B", type=int, default=200)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--dpi", type=int, default=300)
    args = ap.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    df, columns, stats = load_counts(args.scan)
    col = {k: i for i, k in enumerate(columns)}
    n_docs = int(df["docs"].sum())
    n_bl = int(df["ex_blocklist"].sum())
    n_tx = int(df["ex_toxic"].sum())
    n_both = int(df["both_ex"].sum())
    n_bl_only = int(df["blocklist_only"].sum())
    n_tx_only = int(df["toxic_only"].sum())
    n_kept_both = int(df["both_kept"].sum())

    boot = bootstrap(df[columns].to_numpy(), args.B, args.seed)
    b_docs = boot[:, col["docs"]]
    b_bl = boot[:, col["ex_blocklist"]]
    b_tx = boot[:, col["ex_toxic"]]

    # 1. Identity Term PMI comparison
    rows_pmi = []
    for term, cat in IDENTITY:
        bl_docs = df[f"bl:{term}|docs"].sum()
        bl_ex = df[f"bl:{term}|excluded"].sum()
        tx_ex = df[f"tx:{term}|excluded"].sum()

        with np.errstate(divide="ignore", invalid="ignore"):
            bl_pmi = np.log(bl_ex * n_docs / (bl_docs * n_bl)) if (bl_docs and bl_ex) else np.nan
            tx_pmi = np.log(tx_ex * n_docs / (bl_docs * n_tx)) if (bl_docs and tx_ex) else np.nan

            b_bl_pmi = np.log(boot[:, col[f"bl:{term}|excluded"]] * b_docs / (boot[:, col[f"bl:{term}|docs"]] * b_bl))
            b_tx_pmi = np.log(boot[:, col[f"tx:{term}|excluded"]] * b_docs / (boot[:, col[f"tx:{term}|docs"]] * b_tx))

        bl_lo, bl_hi = ci(b_bl_pmi[np.isfinite(b_bl_pmi)]) if np.isfinite(b_bl_pmi).sum() > 10 else (np.nan, np.nan)
        tx_lo, tx_hi = ci(b_tx_pmi[np.isfinite(b_tx_pmi)]) if np.isfinite(b_tx_pmi).sum() > 10 else (np.nan, np.nan)

        rows_pmi.append({
            "term": term,
            "category": cat,
            "total_docs": int(bl_docs),
            "bl_excluded": int(bl_ex),
            "bl_removal_%": 100 * bl_ex / max(1, bl_docs),
            "bl_pmi": bl_pmi,
            "bl_ci_lo": bl_lo,
            "bl_ci_hi": bl_hi,
            "tx_excluded": int(tx_ex),
            "tx_removal_%": 100 * tx_ex / max(1, bl_docs),
            "tx_pmi": tx_pmi,
            "tx_ci_lo": tx_lo,
            "tx_ci_hi": tx_hi,
        })

    df_pmi = pd.DataFrame(rows_pmi).sort_values("bl_pmi", ascending=False).reset_index(drop=True)
    df_pmi.round(3).to_csv(os.path.join(args.out_dir, "pmi_comparison.csv"), index=False)

    # Plot Figure: PMI Comparison
    fig, ax = plt.subplots(figsize=(10, 8), facecolor=SURFACE)
    style(ax)
    y_positions = np.arange(len(df_pmi))
    h = 0.35

    ax.barh(y_positions + h / 2, df_pmi["bl_pmi"], height=h, color=C_BL, label="Blocklist (LDNOOBW regex)", alpha=0.9)
    ax.barh(y_positions - h / 2, df_pmi["tx_pmi"], height=h, color=C_TX, label="Toxicity Classifier (Context-aware)", alpha=0.9)

    ax.set_yticks(y_positions)
    ax.set_yticklabels(df_pmi["term"], fontsize=9, color=INK)
    ax.invert_yaxis()
    ax.set_xlabel("Pointwise Mutual Information (PMI) with Exclusion", color=INK2, fontsize=10)
    ax.set_title("Exclusion Disparity: Blocklist vs. Toxicity Classifier\n"
                 "Higher PMI = term is discarded disproportionately more often than baseline",
                 color=INK, loc="left", fontsize=11, weight="bold")
    ax.legend(frameon=False, loc="lower right", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out_dir, "fig_pmi_comparison.png"), dpi=args.dpi, facecolor=SURFACE)
    plt.close(fig)

    # 2. Dialect comparison
    rows_dial = []
    for d in DIALECTS:
        n_d = df[f"bl:dialect:{d}|docs"].sum()
        bl_d_ex = df[f"bl:dialect:{d}|excluded"].sum()
        tx_d_ex = df[f"tx:dialect:{d}|excluded"].sum()

        b_d_docs = boot[:, col[f"bl:dialect:{d}|docs"]]
        b_d_bl_ex = boot[:, col[f"bl:dialect:{d}|excluded"]]
        b_d_tx_ex = boot[:, col[f"tx:dialect:{d}|excluded"]]

        with np.errstate(divide="ignore", invalid="ignore"):
            bl_rate = 100 * bl_d_ex / max(1, n_d)
            tx_rate = 100 * tx_d_ex / max(1, n_d)

            b_bl_rate = 100 * b_d_bl_ex / np.maximum(1, b_d_docs)
            b_tx_rate = 100 * b_d_tx_ex / np.maximum(1, b_d_docs)

        bl_lo, bl_hi = ci(b_bl_rate)
        tx_lo, tx_hi = ci(b_tx_rate)

        rows_dial.append({
            "dialect": d,
            "total_docs": int(n_d),
            "bl_excluded": int(bl_d_ex),
            "bl_removal_%": bl_rate,
            "bl_ci_lo": bl_lo,
            "bl_ci_hi": bl_hi,
            "tx_excluded": int(tx_d_ex),
            "tx_removal_%": tx_rate,
            "tx_ci_lo": tx_lo,
            "tx_ci_hi": tx_hi,
        })

    df_dial = pd.DataFrame(rows_dial)
    df_dial.round(3).to_csv(os.path.join(args.out_dir, "dialects_comparison.csv"), index=False)

    # Plot Figure: Dialects Comparison
    fig, ax = plt.subplots(figsize=(8, 4.5), facecolor=SURFACE)
    style(ax)
    valid_dial = [d for d in DIALECTS if d != "unassigned"]
    df_vd = df_dial[df_dial["dialect"].isin(valid_dial)]
    x_positions = np.arange(len(df_vd))
    w = 0.35

    ax.bar(x_positions - w / 2, df_vd["bl_removal_%"], width=w, color=C_BL, label="Blocklist (LDNOOBW)", alpha=0.9)
    ax.bar(x_positions + w / 2, df_vd["tx_removal_%"], width=w, color=C_TX, label="Toxicity Classifier", alpha=0.9)

    ax.set_xticks(x_positions)
    ax.set_xticklabels(df_vd["dialect"], fontsize=10, color=INK)
    ax.set_ylabel("Removal Rate (%)", color=INK2, fontsize=10)
    ax.set_title("Dialect Removal Rate: Blocklist vs. Toxicity Classifier\n"
                 "Shows whether neural classifier mitigates the 5x censorship penalty on AAE",
                 color=INK, loc="left", fontsize=11, weight="bold")
    ax.legend(frameon=False, loc="upper right", fontsize=9)
    fig.tight_layout()
    fig.savefig(os.path.join(args.out_dir, "fig_dialects_comparison.png"), dpi=args.dpi, facecolor=SURFACE)
    plt.close(fig)

    # 3. Discrepancy samples compilation
    disc_files = sorted(glob.glob(os.path.join(args.scan, "discrepancies_*.jsonl.gz")))
    sample_rows = []
    for p in disc_files:
        with gzip.open(p, "rt", encoding="utf-8") as g:
            for line in g:
                sample_rows.append(json.loads(line))
                if len(sample_rows) >= 500:
                    break
        if len(sample_rows) >= 500:
            break

    if sample_rows:
        pd.DataFrame(sample_rows).to_csv(os.path.join(args.out_dir, "inspect_discrepancies.csv"), index=False)

    # 4. Generate summary.md
    with open(os.path.join(args.out_dir, "summary.md"), "w", encoding="utf-8") as f:
        f.write("# Arm B Audit: Blocklist (LDNOOBW) vs. Toxicity Classifier\n\n")
        f.write(f"- Total documents evaluated: **{n_docs:,}** across **{len(df):,} web domains**\n")
        f.write(f"- Overall Blocklist exclusion rate: **{100 * n_bl / max(1, n_docs):.2f}%**\n")
        f.write(f"- Overall Toxicity Classifier exclusion rate: **{100 * n_tx / max(1, n_docs):.2f}%**\n\n")
        f.write("## 1. Filter Decision Agreement Matrix\n\n")
        f.write("| Category | Documents | Share of Total | Meaning |\n")
        f.write("|---|---|---|---|\n")
        f.write(f"| **Both Kept** | {n_kept_both:,} | {100 * n_kept_both / max(1, n_docs):.2f}% | Clean text accepted by both filters |\n")
        f.write(f"| **Both Excluded** | {n_both:,} | {100 * n_both / max(1, n_docs):.2f}% | Unambiguous toxicity/obscenity caught by both |\n")
        f.write(f"| **Blocklist Only** | {n_bl_only:,} | {100 * n_bl_only / max(1, n_docs):.2f}% | **Blocklist False Alarms**: benign text censored by crude keywords |\n")
        f.write(f"| **Toxicity Only** | {n_tx_only:,} | {100 * n_tx_only / max(1, n_docs):.2f}% | **Blocklist False Negatives**: toxic text missing from the static word list |\n\n")
        f.write("## 2. Top Identity Terms: Reduction in Censorship Penalty\n\n")
        f.write("| Identity Term | Category | Docs | Blocklist Excluded (%) | Classifier Excluded (%) | Delta (% pts) |\n")
        f.write("|---|---|---|---|---|---|\n")
        for r in df_pmi.head(10).itertuples():
            f.write(f"| **{r.term}** | {r.category} | {r.total_docs:,} | {r.bl_removal_:.1f}% | {r.tx_removal_:.1f}% | {r.tx_removal_ - r.bl_removal_:.1f}% |\n")
        f.write("\n## 3. Dialect Removal Comparison\n\n")
        f.write("| Dialect | Total Docs | Blocklist Removal (%) | Classifier Removal (%) |\n")
        f.write("|---|---|---|---|\n")
        for r in df_dial.itertuples():
            f.write(f"| **{r.dialect}** | {r.total_docs:,} | {r.bl_removal_:.2f}% | {r.tx_removal_:.2f}% |\n")

    print(f"Analysis complete! Results written to {args.out_dir}", flush=True)


if __name__ == "__main__":
    main()


## 3. Run Side-by-Side Scan (Blocklist vs. Toxicity Classifier)

Scans 4 seeded shards of `en.noblocklist`. Both the LDNOOBW regex and `unitary/toxic-bert`
score each document simultaneously. Resumable!


In [ ]:
%%bash
cd /kaggle/working/f5_toxicity
python scripts/scan_toxicity.py --model unitary/toxic-bert --threshold 0.5 \
    --n 4 --seed 0 --batch_size 128 --max_length 256 --out_dir /kaggle/working/f5_toxicity/results/arm_b_toxicity_4_shards


## 4. Run Comparative Statistical Analysis & Generate Plots


In [ ]:
%%bash
cd /kaggle/working/f5_toxicity
python scripts/analyze_toxicity.py --scan /kaggle/working/f5_toxicity/results/arm_b_toxicity_4_shards --out_dir /kaggle/working/f5_toxicity/results/arm_b_toxicity_4_shards/analysis --B 200


## 5. Review Findings & Publications Plots


In [ ]:
import os
import pandas as pd
from IPython.display import Image, Markdown, display

A = "/kaggle/working/f5_toxicity/results/arm_b_toxicity_4_shards/analysis"
show = lambda p, w=950: display(Image(p, width=w)) if os.path.exists(p) else print("Missing:", p)

if os.path.exists(f"{A}/summary.md"):
    display(Markdown(open(f"{A}/summary.md").read()))

show(f"{A}/fig_pmi_comparison.png")
show(f"{A}/fig_dialects_comparison.png")

if os.path.exists(f"{A}/pmi_comparison.csv"):
    display(Markdown("### Identity Term Exclusion Comparison (Top 10)"))
    display(pd.read_csv(f"{A}/pmi_comparison.csv").head(10))

if os.path.exists(f"{A}/dialects_comparison.csv"):
    display(Markdown("### Dialect Removal Rate Comparison"))
    display(pd.read_csv(f"{A}/dialects_comparison.csv"))


## 6. Package and Download Results Zip


In [ ]:
%%bash
cd /kaggle/working
zip -qr f5_arm_b_toxicity_results.zip f5_toxicity/results
ls -lh f5_arm_b_toxicity_results.zip
